# Ceasul Cuantic al Iașului
## Nivelul 6 — Clopotele

> ### Înapoi în turnul Palatului Culturii, ora 14:30
>
> Aveți cheia de repornire. Acum o folosiți pe **adevăratul QIași-1**: un procesor cuantic IBM, răcit aproape de zero absolut.
>
> Aici nu mai e simulare. Fiecare qubit e un mic circuit supraconductor, iar fiecare are defectele lui.
>
> Și tot aici o veți întâlni, în sfârșit, pe Umbra.

### Ce veți învăța în acest nivel

- cum rulați un circuit pe un **calculator cuantic real** IBM;
- de ce rezultatele reale au **zgomot** și cum alegeți cei mai buni qubiți;
- cum **corectați** erorile de citire rezolvând un **sistem de ecuații liniare** (clasa a XI-a!).

**Puteți începe părțile A și B imediat după nivelul 3**: circuitul așteaptă la coadă cât lucrați la nivelurile 4 și 5. Bugetul vostru este de cel mult 2–3 joburi pe hardware.

In [ ]:
# Rulați această celulă prima. Ea încarcă Qiskit și uneltele provocării.
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from ceas_cuantic import *

ECHIPA = "Numele echipei"   # COMPLETEAZĂ AICI — același nume în toate notebook-urile
COD_ECHIPA = "XXXX-XXX"     # COMPLETEAZĂ AICI — codul primit de la organizatori
start(ECHIPA, COD_ECHIPA)

## 0. Contul IBM Quantum

Mentorul vostru a creat contul echipei pe [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com/). Dacă nu e salvat deja pe acest calculator, cereți-i cheia API și instanța (CRN), decomentați liniile de mai jos și rulați celula **o singură dată**. Apoi comentați-le la loc, ca cheia să nu rămână în fișier.

Dacă nu există cont sau coada e foarte lungă, celula alege automat un **simulator cu zgomot** construit după un procesor IBM real.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# QiskitRuntimeService.save_account(token="<CHEIA_API>", instance="<CRN>", overwrite=True, set_as_default=True)


def simulator_cu_zgomot():
    from qiskit_ibm_runtime import fake_provider
    for nume in ["FakeTorino", "FakeFez", "FakeBrisbane", "FakeSherbrooke", "FakeManilaV2"]:
        if hasattr(fake_provider, nume):
            return getattr(fake_provider, nume)()


try:
    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
    print("Calculator cuantic real:", backend.name)
except Exception as e:
    print("Nu am găsit un cont IBM Quantum salvat:", e)
    backend = simulator_cu_zgomot()
    print("Folosim simulatorul cu zgomot:", backend.name)

## 1. Zgomotul

Pe un procesor real apar trei feluri de erori:

- **erori de poartă** — fiecare poartă e un impuls de microunde puțin imperfect;
- **decoerență** — qubitul își pierde treptat starea, din cauza mediului (căldură, vibrații, radiație);
- **erori de citire** — la final, aparatul citește uneori 1 în loc de 0 sau invers.

Fiecare qubit fizic are erorile lui, măsurate zilnic de IBM. Mai mult, nu toți qubiții sunt legați direct între ei: CNOT se poate aplica doar pe perechi conectate (harta se numește *coupling map*).

<div class="alert alert-block alert-success">
<b>Sarcina 6A — Cei mai buni qubiți (80 de puncte)</b>

Găsiți perechea de qubiți <b>conectați direct</b> cu cea mai mică <b>sumă a erorilor de citire</b>.<br>
<code>backend.coupling_map.get_edges()</code> dă lista perechilor conectate, iar <code>eroare_citire(q)</code> dă eroarea de citire a qubitului q.
</div>

In [ ]:
def eroare_citire(q):
    e = backend.target["measure"][(q,)].error
    return e if e is not None else 1.0


pereche = backend.coupling_map.get_edges()[0]   # o primă pereche; bucla voastră o va înlocui
cea_mai_mica = 1.0
for (a, b) in backend.coupling_map.get_edges():
    pass   # COMPLETEAZĂ AICI

print(f"Perechea aleasă: {pereche}, suma erorilor de citire: {cea_mai_mica:.4f}")
verifica_6A(pereche, backend)

Procesorul nu știe direct porțile H și CNOT: circuitul trebuie **transpilat**, adică rescris cu porțile native ale procesorului, pe qubiții aleși. Comparați un nivel de optimizare mic cu unul mare:

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

for nivel in (0, 3):
    pm = generate_preset_pass_manager(optimization_level=nivel, backend=backend, initial_layout=list(pereche))
    isa = pm.run(bell)
    print(f"optimizare {nivel}: adâncime {isa.depth()}, porți pe 2 qubiți: {isa.num_nonlocal_gates()}")
display(isa.draw("mpl", idle_wires=False))

## 2. Rularea pe hardware

Într-un singur job trimitem 5 circuite: perechea Bell și 4 **circuite de calibrare**, care pregătesc pe rând stările cunoscute 00, 01, 10, 11. Din ele vom afla cât de des greșește aparatul la citire.

<div class="alert alert-block alert-success">
<b>Sarcina 6B — Rulare pe un calculator cuantic real (100 de puncte · clopotul 7)</b>

Completați <code>circuit_calibrare(stare)</code>: pentru <code>stare</code> = "01", de exemplu, qubitul 0 trebuie să fie 1, iar qubitul 1 să fie 0 (citim de la dreapta la stânga!).<br>
Apoi trimiteți jobul și notați-i ID-ul. Clopotul 7 se aprinde doar pe un procesor real.
</div>

In [ ]:
def circuit_calibrare(stare):
    qc = QuantumCircuit(2)
    # COMPLETEAZĂ AICI: X pe qubiții care trebuie să fie 1
    qc.measure_all()
    return qc


stari = ["00", "01", "10", "11"]
circuite = [bell] + [circuit_calibrare(s) for s in stari]

pm = generate_preset_pass_manager(optimization_level=1, backend=backend, initial_layout=list(pereche))
isa_circuite = pm.run(circuite)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuite, shots=4000)
print("ID-ul jobului (notați-l!):", job.job_id())

Celula următoare așteaptă rezultatul. Dacă ați închis notebook-ul între timp, recuperați jobul după ID cu linia comentată.

In [ ]:
# job = service.job("ID-UL-JOBULUI")

rezultat = job.result()
counts_bell = rezultat[0].data.meas.get_counts()
counts_cal = {s: rezultat[i + 1].data.meas.get_counts() for i, s in enumerate(stari)}

print("Bell pe hardware:", counts_bell)
plot_histogram([masoara(bell, shots=4000), counts_bell], legend=["simulator ideal", backend.name])

In [ ]:
verifica_6B(counts_bell, backend)
# Dacă ați folosit simulatorul cu zgomot din cauza cozii, mentorul vă poate da un cod de rezervă:
# verifica_6B(counts_bell, backend, cod_mentor="...")

## 3. Corectarea erorilor de citire

**Întâi pe hârtie, cu un singur qubit.** Calibrarea arată că, atunci când qubitul e pregătit în 0, aparatul citește 0 în 97% din cazuri (și 1 în 3%). Când e pregătit în 1, citește 1 în 92% din cazuri (și 0 în 8%).

Pe o stare necunoscută, aparatul citește 0 în 55% din cazuri și 1 în 45%. Care sunt probabilitățile **adevărate** $p_0, p_1$?

$$\begin{pmatrix} 0{,}97 & 0{,}08 \\ 0{,}03 & 0{,}92 \end{pmatrix} \begin{pmatrix} p_0 \\ p_1 \end{pmatrix} = \begin{pmatrix} 0{,}55 \\ 0{,}45 \end{pmatrix}$$

Matricea $M$ se numește **matricea de confuzie**: coloana $j$ spune ce citește aparatul când starea adevărată este $j$. Rezolvați sistemul cu **regula lui Cramer**.

In [ ]:
p0 = 0   # COMPLETEAZĂ AICI (puteți face calculul în Python)
p1 = 0
verifica_6C_hartie(p0, p1)

### Cu doi qubiți și date reale

Cu doi qubiți, matricea de confuzie devine $4\times4$: coloana $j$ conține ce citește aparatul când ați pregătit starea $j$ (00, 01, 10 sau 11) — exact ce au măsurat circuitele voastre de calibrare. Sistemul are acum 4 necunoscute; în loc de Cramer, îl rezolvă calculatorul.

Funcția gata făcută `corecteaza_citirea(counts_cal, counts_bell)` construiește $M$, rezolvă sistemul $M \cdot p_{corectat} = p_{măsurat}$ și returnează `M`, `p_masurat` și `p_corectat` (în ordinea 00, 01, 10, 11).

<div class="alert alert-block alert-success">
<b>Sarcina 6C — Zgomotul corectat (120 de puncte · clopotul 8)</b>

<ol>
<li>Rulați corecția și uitați-vă la matricea M: unde sunt numerele mari? Ce înseamnă numerele mici din afara diagonalei?</li>
<li>Calculați fidelitatea perechii Bell, P(00) + P(11), <b>înainte</b> (<code>f_inainte</code>) și <b>după</b> corecție (<code>f_dupa</code>).</li>
</ol>
</div>

In [ ]:
M, p_masurat, p_corectat = corecteaza_citirea(counts_cal, counts_bell)

print("Matricea de confuzie M (coloane = starea pregătită 00, 01, 10, 11):\n", np.round(M, 3))
print("măsurat:  ", np.round(p_masurat, 3))
print("corectat: ", np.round(p_corectat, 3))

f_inainte = 0   # COMPLETEAZĂ AICI: P(00) + P(11) din p_masurat
f_dupa = 0      # COMPLETEAZĂ AICI: P(00) + P(11) din p_corectat

In [ ]:
plt.bar(np.arange(4) - 0.2, p_masurat, width=0.4, label="măsurat")
plt.bar(np.arange(4) + 0.2, p_corectat, width=0.4, label="corectat")
plt.xticks(range(4), stari)
plt.ylabel("probabilitate")
plt.legend()
plt.show()

verifica_6C(M, p_corectat, counts_bell, f_inainte, f_dupa)

**Discutați:** corecția a reparat doar erorile de **citire**. Ce erori au rămas? (Gândiți-vă la lista de la începutul secțiunii 1.)

<div class="alert alert-block alert-warning">
<b>Provocare extra — Grover pe hardware (+50 de puncte)</b>

Rulați circuitul <code>grover_8(2)</code> de la nivelul 5 pe hardware (aveți nevoie de 3 qubiți conectați în linie), împreună cu 8 circuite de calibrare pentru stările 000…111.<br>
Corectați rezultatul cu <code>corecteaza_citirea</code> (merge și cu 8 stări) și comparați probabilitatea locului căutat înainte (<code>p_brut</code>) și după corecție (<code>p_corectat_grover</code>).<br>
Cereți-i mentorului acordul înainte de a trimite jobul: consumă din bugetul de timp.
</div>

In [ ]:
# COMPLETEAZĂ AICI

p_brut = 0
p_corectat_grover = 0
verifica_extra6(p_brut, p_corectat_grover)

<div class="alert alert-block alert-info">
<b>La AC studiezi asta</b>

Ați construit un <b>model al zgomotului</b> (matricea M) din măsurători și l-ați folosit ca să corectați datele. Asta înseamnă <b>identificarea sistemelor</b>, o disciplină de bază la Automatică, și <b>prelucrarea semnalelor</b>.<br>
Aceleași idei corectează senzorii unui robot, imaginile unei camere sau semnalul unui electrocardiograf.
</div>

> ### Ora 16:00. Turnul Palatului Culturii
>
> Pe ecranul lui QIași-1 apare un ultim mesaj: **UMBRA = DECOERENȚĂ**.
>
> Umbra nu a fost niciodată o persoană. Este **zgomotul** — căldura, vibrațiile, radiația — care atinge fiecare qubit real și îi strică încet starea. Nimeni nu o poate învinge de tot.
>
> Dar ați făcut exact ce fac inginerii: ați măsurat-o, i-ați construit un model și ați corectat-o.
>
> Clopotele pornesc. Opt clopote, *Hora Unirii*, peste tot Iașul.

## Finalul provocării

Rulați celula de mai jos. Dacă toate cele 8 clopote sunt aprinse, chemați-l pe mentor: este momentul pentru *Hora Unirii*.

Pregătiți acum prezentarea de 5 minute: un concept explicat pe înțelesul unui elev de clasa a IX-a, ce ați făcut și ce n-a mers, și o idee despre cum ar putea ajuta calculul cuantic Iașul.

In [ ]:
turn()